In [ ]:
import os
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

# Pour affichage plus joli
import matplotlib
matplotlib.rcParams['image.cmap'] = 'gray'

group_outputs = {
    'Groupe1': 'Tollsome_output1',
    'Groupe2': 'Tollsome_output2',
    'Groupe3': 'Tollsome_output3'
}

# Dossiers de base
base_image_dir = '/home/amenacer/Stage/Data/Segmentation/Tollsome'
base_mask_dir = '/home/amenacer/Stage/Data/Segmentation/Tollsomemask'
base_output_dir = '/home/amenacer/Stage/Data/Segmentation/resultas'

for group_name, output_folder in group_outputs.items():
    print(f"\n🔁 Traitement de {group_name}")

    images_dir = os.path.join(base_image_dir, group_name)
    masks_dir = os.path.join(base_mask_dir, group_name)
    output_dir = os.path.join(base_output_dir, output_folder)
    os.makedirs(output_dir, exist_ok=True)

    for filename in os.listdir(images_dir):
        if filename.endswith('.nii.gz'):
            image_path = os.path.join(images_dir, filename)
            mask_path = os.path.join(masks_dir, filename.replace('_0000', ''))
            output_path = os.path.join(output_dir, filename)

            if not os.path.exists(mask_path):
                print(f"⚠️ Masque manquant pour : {filename}")
                continue

            # Chargement
            img_nii = nib.load(image_path)
            mask_nii = nib.load(mask_path)

            img = img_nii.get_fdata()
            mask = mask_nii.get_fdata()

            if img.shape != mask.shape:
                print(f"❌ Dimensions incompatibles pour : {filename}")
                continue

            # Application
            applied = img * mask

            # Sauvegarde
            applied_nii = nib.Nifti1Image(applied, affine=img_nii.affine)
            nib.save(applied_nii, output_path)

            print(f"✅ {filename} traité dans {group_name}")

            # === Visualisation ===
            slice_idx = img.shape[2] // 2  # Coupe centrale

            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            axes[0].imshow(img[:, :, slice_idx])
            axes[0].set_title('Image originale')

            axes[1].imshow(mask[:, :, slice_idx], cmap='Reds')
            axes[1].set_title('Masque')

            axes[2].imshow(applied[:, :, slice_idx])
            axes[2].set_title('Image × Masque')

            for ax in axes:
                ax.axis('off')
            plt.tight_layout()
            plt.show()
